# STAF — Colab Launcher

This notebook is just a thin wrapper. All the actual model code lives in the repo:

- **Training loop** → [`train.py`](https://github.com/RAJ-VARUN13/staf-forensics-multimodal/blob/main/train.py)
- **Evaluation** → [`evaluate.py`](https://github.com/RAJ-VARUN13/staf-forensics-multimodal/blob/main/evaluate.py)
- **Models / datasets / preprocessing** → `staf/`

You can safely **Run All** (`Ctrl+F9`).  
Re-running is safe — clone and unzip steps are idempotent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO = 'staf-forensics-multimodal'
if not os.path.isdir(f'/content/{REPO}'):
    !git clone https://github.com/RAJ-VARUN13/{REPO}.git
else:
    print(f'{REPO}/ already exists, pulling latest...')
    !git -C /content/{REPO} pull --ff-only

%cd /content/{REPO}
!pip install -q -r requirements.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type and pick a GPU.'
    )

print(f'GPU ready: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Automatically locate the FakeAVCeleb dataset (unzipped folder or ZIP file) on Google Drive
import os
import glob
import shutil

target_path = '/content/FakeAVCeleb'
dataset_linked = False

if os.path.exists(target_path):
    print(f'Dataset already exists at {target_path}')
    dataset_linked = True
else:
    # 1. Search for unzipped folders on mounted Google Drive
    search_dirs = [
        '/content/drive/MyDrive',
        '/content/drive/Shareddrives'
    ]
    
    print('Searching for dataset directory on Google Drive...')
    for parent in search_dirs:
        if not os.path.exists(parent):
            continue
        for root, dirs, files in os.walk(parent):
            # Check if this folder has Real and Fake subfolders
            has_fake = any('fake' in d.lower() for d in dirs)
            has_real = any('real' in d.lower() for d in dirs)
            if has_fake and has_real:
                found_dir = root
                # Exclude experiment/results or processed folders
                if 'results' not in found_dir.lower() and 'processed' not in found_dir.lower():
                    print(f'Found matching dataset directory: {found_dir}')
                    os.symlink(found_dir, target_path)
                    print(f'Created symlink: {target_path} -> {found_dir}')
                    dataset_linked = True
                    break
        if dataset_linked:
            break

    # 2. If no folder found, search for zip file
    if not dataset_linked:
        print('Searching for dataset ZIP file on Google Drive...')
        zip_files = []
        for parent in search_dirs:
            if not os.path.exists(parent):
                continue
            for root, dirs, files in os.walk(parent):
                for f in files:
                    if f.endswith('.zip') and ('fakeav' in f.lower() or 'multimodal' in f.lower()):
                        zip_files.append(os.path.join(root, f))
        
        if zip_files:
            zip_path = zip_files[0]
            print(f'Found dataset ZIP file: {zip_path}')
            print('Extracting dataset...')
            # Run unzip shell command via IPython API
            from IPython import get_ipython
            get_ipython().system(f'unzip -qn "{zip_path}" -d /content/')
            
            # Check if extracted folder is named FakeAVCeleb or similar
            extracted_dirs = [d for d in os.listdir('/content') if os.path.isdir(os.path.join('/content', d)) and 'fakeav' in d.lower()]
            if extracted_dirs and not os.path.exists(target_path):
                os.rename(os.path.join('/content', extracted_dirs[0]), target_path)
            dataset_linked = True
        else:
            raise FileNotFoundError(
                'Could not locate FakeAVCeleb folder or ZIP file on your Google Drive.\n'
                'Please upload the dataset folder or FakeAVCeleb_v1.2.zip to your Google Drive.'
            )

if dataset_linked:
    # Double check directories in /content/FakeAVCeleb
    print('Dataset contents:', os.listdir(target_path))


In [ ]:
CONFIG = 'staf/configs/colab.yaml'

!python -m staf.preprocessing.extract_frames  --config {CONFIG}
!python -m staf.preprocessing.detect_faces    --config {CONFIG}
!python -m staf.preprocessing.crop_faces      --config {CONFIG}
!python -m staf.preprocessing.extract_audio   --config {CONFIG}
!python -m staf.preprocessing.sync_modalities --config {CONFIG}

In [ ]:
!python train.py --config {CONFIG} --no_wandb --overrides training.max_epochs=1

In [ ]:
import glob

checkpoints = sorted(glob.glob('/content/results/*/checkpoints/best_model.pt'))
if not checkpoints:
    raise FileNotFoundError(
        'No best_model.pt found under /content/results/. '
        'Did training finish or get interrupted?'
    )

ckpt = checkpoints[-1]   # latest run
print(f'Evaluating: {ckpt}')
!python evaluate.py --checkpoint "{ckpt}" --split test

In [ ]:
import os, shutil

DRIVE_DEST = '/content/drive/MyDrive/STAF_experiments'
os.makedirs(DRIVE_DEST, exist_ok=True)

!cp -ru /content/results/* "{DRIVE_DEST}/"
print(f'Results backed up to {DRIVE_DEST}')